In [1]:
import os
import math
import random
from glob import glob
from typing import List, Tuple, Optional
import numpy as np
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontextm

# =========================
# CONFIG
# =========================
TRAIN_IMG_DIR = "/kaggle/input/udids2025/UDIADS/images/train"
TRAIN_MSK_DIR = "/kaggle/input/udids2025/UDIADS/masks/train"
VAL_IMG_DIR = "/kaggle/input/udids2025/UDIADS/images/val"
VAL_MSK_DIR = "/kaggle/input/udids2025/UDIADS/masks/val"

OUT_DIR = "/kaggle/working/"
CKPT_BEST = os.path.join(OUT_DIR, "best_maskprob_gauss.pt")
CKPT_LAST = os.path.join(OUT_DIR, "last_maskprob_gauss.pt")
LOG_CSV = os.path.join(OUT_DIR, "metrics_maskprob_gauss.csv")
PREVIEW_DIR = os.path.join(OUT_DIR, "preview_maskprob_gauss")

RUN_TRAIN = True
RUN_INFER = False   # set True to preview on val set after training

EPOCHS = 500
BATCH_SIZE = 1
BASE_CHANNELS = 32
LR_INIT = 1e-4
WEIGHT_DECAY = 0.0

# DataLoader perf (Windows-safe: keep 0 to avoid multiprocess worker issues)
NUM_WORKERS = 0
PIN_MEMORY = True
PERSISTENT = False  # ignored when NUM_WORKERS=0
PREFETCH = None   # ignored when NUM_WORKERS=0

# Hardware knobs
USE_AMP = False
USE_COMPILE = False
# image is downscaled so the longer side <= MAX_SIDE (aspect preserved)
MAX_SIDE = 768

# Oriented Gaussian GT settings
RX_SCALE = 1.8
RY_SCALE = 1.2
ROI_SIGMA = 3.0
USE_CACHE_GT = True
# We cache the ORIGINAL-RES Gaussian here:
CACHE_DIR = os.path.join(OUT_DIR, "gt_cache_maskprob_gauss_orig")

# Loss choice: "mse" (recommended) or "bce" or "mse+bce"
GAUSS_LOSS = "mse"
BCE_WEIGHT = 0.3
GAUSS_SSIM_WEIGHT = 0.0

# Visualization
PRED_THR = 0.3

SEED = 42
ALLOWED_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
PREVIEW_EVERY = 25

# =========================
# Utils / I/O
# =========================


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def load_gray(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))


def load_mask_0255(path: str) -> np.ndarray:
    arr = np.array(Image.open(path).convert("L"))
    return (arr > 0).astype(np.uint8) * 255


def save_png(arr: np.ndarray, path: str):
    d = os.path.dirname(path)
    os.makedirs(d, exist_ok=True) if d else None
    Image.fromarray(arr).save(path)


def _resize_keep_aspect_img(img_np: np.ndarray, max_side: int = MAX_SIDE):
    """Resize single image so its longer side <= max_side (aspect preserved)."""
    H, W = img_np.shape
    s = max(H, W)
    if s <= max_side:
        return img_np, 1.0
    scale = max_side / float(s)
    newW = int(round(W * scale))
    newH = int(round(H * scale))
    img_res = np.array(Image.fromarray(
        img_np).resize((newW, newH), Image.BILINEAR))
    return img_res, scale

# =========================
# Oriented Gaussian GT (computed at ORIGINAL resolution)
# =========================


def build_oriented_gauss_from_mask(mask_0255: np.ndarray):
    H, W = mask_0255.shape
    fg = (mask_0255 > 0).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(
        fg, connectivity=4)
    g_all = np.zeros((H, W), np.float32)
    if num <= 1:
        return g_all

    for k in range(1, num):
        x, y, w, h, area = stats[k]
        if area < 1:
            continue
        roi_lbl = labels[y:y+h, x:x+w]
        roi_msk = (roi_lbl == k).astype(np.uint8)

        m = cv2.moments(roi_msk, binaryImage=True)
        m00 = m["m00"]
        if m00 <= 0:
            continue

        mx = float(m["m10"] / m00) + x
        my = float(m["m01"] / m00) + y

        cxx = float(m["mu20"] / m00)
        cyy = float(m["mu02"] / m00)
        cxy = float(m["mu11"] / m00)
        trace = cxx + cyy
        det = cxx * cyy - cxy * cxy
        tmp = max(trace*trace/4 - det, 0.0) ** 0.5
        lam1 = trace/2 + tmp
        lam2 = max(trace/2 - tmp, 1e-12)

        std_major = max(lam1, 1e-12) ** 0.5
        std_minor = max(lam2, 1e-12) ** 0.5
        rx = max(2.0, RX_SCALE * std_major)
        ry = max(2.0, RY_SCALE * std_minor)

        theta = 0.5 * math.atan2(2*cxy, (cxx - cyy))
        c, s = math.cos(theta), math.sin(theta)

        # Limit ROI ~ 3σ (speed)
        hx = ROI_SIGMA * math.sqrt((rx*c)**2 + (ry*(-s))**2)
        hy = ROI_SIGMA * math.sqrt((rx*s)**2 + (ry*(c))**2)
        x0 = max(0, int(math.floor(mx - hx)))
        x1 = min(W-1, int(math.ceil(mx + hx)))
        y0 = max(0, int(math.floor(my - hy)))
        y1 = min(H-1, int(math.ceil(my + hy)))
        if x1 < x0 or y1 < y0:
            continue

        xx = np.arange(x0, x1+1, dtype=np.float32)[None, :]
        yy = np.arange(y0, y1+1, dtype=np.float32)[:, None]
        dx, dy = xx - mx, yy - my
        t = dx * c + dy * s
        v = -dx * s + dy * c
        g = np.exp(-0.5 * ((t/(rx+1e-6))**2 + (v/(ry+1e-6))**2)
                   ).astype(np.float32)

        cur = g_all[y0:y1+1, x0:x1+1]
        np.maximum(cur, g, out=cur)
        g_all[y0:y1+1, x0:x1+1] = cur

    return g_all  # [0,1] float32

# =========================
# Dataset (compute Gaussian at ORIGINAL res, then resize Gaussian only)
# =========================


class MaskProbGaussDataset(Dataset):
    def __init__(self, img_dir: str, msk_dir: str, cache_dir: str = CACHE_DIR):
        assert os.path.isdir(img_dir) and os.path.isdir(
            msk_dir), f"Missing {img_dir} or {msk_dir}"
        os.makedirs(cache_dir, exist_ok=True) if USE_CACHE_GT else None

        img_paths = sorted(p for p in glob(os.path.join(img_dir, "*"))
                           if os.path.splitext(p)[1].lower() in ALLOWED_EXTS)
        msk_paths = sorted(p for p in glob(os.path.join(msk_dir, "*"))
                           if os.path.splitext(p)[1].lower() in ALLOWED_EXTS)
        msk_map = {os.path.splitext(os.path.basename(p))[
            0]: p for p in msk_paths}
        self.pairs = []
        for ip in img_paths:
            key = os.path.splitext(os.path.basename(ip))[0]
            if key in msk_map:
                self.pairs.append((ip, msk_map[key]))
            else:
                print(f"[warn] no mask for {ip}")
        if not self.pairs:
            raise RuntimeError(
                f"No image/mask pairs between {img_dir} and {msk_dir}")

        self.cache_dir = cache_dir
        print(f"[pairs] {len(self.pairs)} from {img_dir} ↔ {msk_dir}")

    def __len__(self): return int(len(self.pairs))

    def __getitem__(self, idx):
        ipath, mpath = self.pairs[idx]
        key = os.path.splitext(os.path.basename(ipath))[0]
        try:
            # --- load ORIGINALS
            img0 = load_gray(ipath)       # H0xW0 uint8
            msk0 = load_mask_0255(mpath)  # H0xW0 0/255

            # --- get (or build) ORIGINAL-RES Gaussian
            if USE_CACHE_GT:
                cpath = os.path.join(self.cache_dir, f"{key}_gauss_orig.npz")
                if os.path.isfile(cpath):
                    data = np.load(cpath)
                    gt_gauss_orig = data["gt_gauss"].astype(np.float32)
                else:
                    gt_gauss_orig = build_oriented_gauss_from_mask(
                        msk0).astype(np.float32)
                    np.savez_compressed(cpath, gt_gauss=gt_gauss_orig)
            else:
                gt_gauss_orig = build_oriented_gauss_from_mask(
                    msk0).astype(np.float32)

            # --- resize IMAGE ONLY for the model
            img_res, _ = _resize_keep_aspect_img(img0, max_side=MAX_SIDE)
            H, W = img_res.shape

            # --- resize ORIGINAL Gaussian to the image-resized domain
            g_res = cv2.resize(gt_gauss_orig, (W, H),
                               interpolation=cv2.INTER_LINEAR)

            # --- tensorize
            x01 = (img_res.astype(np.float32) / 255.0)  # [0,1]
            g_t = torch.from_numpy(g_res)[None, ...]    # 1xHxW (float32)
            x_t = torch.from_numpy(x01)[None, ...]      # 1xHxW

            return {"image": x_t, "gauss": g_t, "size": (H, W), "name": os.path.basename(ipath)}
        except Exception as e:
            raise RuntimeError(
                f"Dataset error on sample '{ipath}' / '{mpath}': {e}") from e


def pad_to_max(batch):
    max_h = max(b["image"].shape[-2] for b in batch)
    max_w = max(b["image"].shape[-1] for b in batch)
    imgs, gauss, sizes, names = [], [], [], []
    for b in batch:
        x, g = b["image"], b["gauss"]
        _, h, w = x.shape
        ph, pw = max_h - h, max_w - w
        if ph or pw:
            x = F.pad(x, (0, pw, 0, ph), value=0.0)
            g = F.pad(g, (0, pw, 0, ph), value=0.0)
        imgs.append(x)
        gauss.append(g)
        sizes.append(b["size"])
        names.append(b["name"])
    return {"image": torch.stack(imgs, 0),
            "gauss": torch.stack(gauss, 0),
            "sizes": sizes, "names": names}

# =========================
# UNet (single mask-prob head)
# =========================


def _gn_groups(C: int) -> int:
    for g in (32, 16, 8, 4, 2, 1):
        if C % g == 0:
            return g
    return 1


class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        g = _gn_groups(out_ch)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(g, out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(g, out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x): return self.net(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x): return self.conv(self.pool(x))


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.reduce = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = F.interpolate(
            x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.reduce(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class UNetMaskProb(nn.Module):
    def __init__(self, in_ch=1, base=64):
        super().__init__()
        self.e1 = DoubleConv(in_ch, base)
        self.e2 = Down(base, base*2)
        self.e3 = Down(base*2, base*4)
        self.e4 = Down(base*4, base*8)
        self.bott = DoubleConv(base*8, base*16)
        self.u4 = Up(base*16, base*8, base*8)
        self.u3 = Up(base*8,  base*4, base*4)
        self.u2 = Up(base*4,  base*2, base*2)
        self.u1 = Up(base*2,  base,   base)
        self.head = nn.Conv2d(base, 1, 1)   # single mask-prob head

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        s4 = self.e4(s3)
        z = self.bott(s4)
        z = self.u4(z, s4)
        z = self.u3(z, s3)
        z = self.u2(z, s2)
        z = self.u1(z, s1)
        return torch.sigmoid(self.head(z))  # [B,1,H,W] in [0,1]

# =========================
# Loss helpers
# =========================


class SimpleSSIM(nn.Module):
    def __init__(self, wt=GAUSS_SSIM_WEIGHT):
        super().__init__()
        self.wt = float(wt)
        size, sigma = 11, 1.5
        pad = size//2
        ax = torch.arange(size, dtype=torch.float32) - size//2
        ker1 = torch.exp(-(ax**2)/(2*sigma*sigma))
        ker1 = ker1/ker1.sum()
        w = torch.outer(ker1, ker1).view(1, 1, size, size)
        self.register_buffer("w", w)
        self.pad = pad

    def forward(self, x, y):
        if self.wt <= 0:
            return x.new_zeros(())
        w = self.w.to(dtype=x.dtype, device=x.device)
        mu_x = F.conv2d(x, w, padding=self.pad)
        mu_y = F.conv2d(y, w, padding=self.pad)
        mu_x2, mu_y2, mu_xy = mu_x*mu_x, mu_y*mu_y, mu_x*mu_y
        sig_x2 = F.conv2d(x*x, w, padding=self.pad) - mu_x2
        sig_y2 = F.conv2d(y*y, w, padding=self.pad) - mu_y2
        sig_xy = F.conv2d(x*y, w, padding=self.pad) - mu_xy
        C1, C2 = 0.01**2, 0.03**2
        ssim = ((2*mu_xy + C1)*(2*sig_xy + C2)) / \
            ((mu_x2 + mu_y2 + C1)*(sig_x2 + sig_y2 + C2))
        return (1.0 - ssim.clamp(0, 1)).mean()


def gaussian_loss(pred_prob, gauss_target):
    if GAUSS_LOSS == "mse":
        return (pred_prob - gauss_target).pow(2).mean()
    elif GAUSS_LOSS == "bce":
        return F.binary_cross_entropy(pred_prob, gauss_target)
    elif GAUSS_LOSS == "mse+bce":
        mse = (pred_prob - gauss_target).pow(2).mean()
        bce = F.binary_cross_entropy(pred_prob, gauss_target)
        return (1.0 - BCE_WEIGHT) * mse + BCE_WEIGHT * bce
    else:
        raise ValueError(f"Unknown GAUSS_LOSS={GAUSS_LOSS}")

# =========================
# CSV + Preview
# =========================


def _append_csv(path, row):
    d = os.path.dirname(path)
    os.makedirs(d, exist_ok=True) if d else None
    import csv
    header_needed = not os.path.isfile(path)
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if header_needed:
            w.writeheader()
        w.writerow(row)


def _save_preview(epoch, x01, g_gt, p_prob):
    os.makedirs(PREVIEW_DIR, exist_ok=True)
    def to_u8(a): return np.clip(a*255.0, 0, 255).astype(np.uint8)
    base = to_u8(x01.squeeze(0).cpu().numpy())
    gt = to_u8(g_gt.squeeze(0).cpu().numpy())
    pr = to_u8(p_prob.squeeze(0).detach().cpu().numpy())
    base3 = cv2.cvtColor(base, cv2.COLOR_GRAY2BGR)
    gt_c = cv2.applyColorMap(gt, cv2.COLORMAP_JET)
    pr_c = cv2.applyColorMap(pr, cv2.COLORMAP_JET)
    ov_gt = cv2.addWeighted(base3, 0.4, gt_c, 0.6, 0)
    ov_pr = cv2.addWeighted(base3, 0.4, pr_c, 0.6, 0)
    mask_bin = (p_prob.squeeze(0).detach().cpu().numpy()
                > PRED_THR).astype(np.uint8) * 255
    cv2.imwrite(os.path.join(PREVIEW_DIR, f"ep{epoch:03d}_img.png"), base3)
    cv2.imwrite(os.path.join(PREVIEW_DIR, f"ep{epoch:03d}_gt_gauss.png"), gt_c)
    cv2.imwrite(os.path.join(PREVIEW_DIR, f"ep{epoch:03d}_pr_prob.png"), pr_c)
    cv2.imwrite(os.path.join(PREVIEW_DIR, f"ep{epoch:03d}_ov_gt.png"), ov_gt)
    cv2.imwrite(os.path.join(PREVIEW_DIR, f"ep{epoch:03d}_ov_pr.png"), ov_pr)
    cv2.imwrite(os.path.join(
        PREVIEW_DIR, f"ep{epoch:03d}_pr_mask_thr{int(PRED_THR*100)}.png"), mask_bin)

# =========================
# Helpers
# =========================


def make_loader(dataset, shuffle):
    kwargs = dict(
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        collate_fn=pad_to_max,
        pin_memory=PIN_MEMORY,
    )
    # Only add mp options if workers > 0
    if NUM_WORKERS > 0:
        kwargs.update(dict(persistent_workers=PERSISTENT,
                      prefetch_factor=(PREFETCH or 2)))
    return DataLoader(dataset, **kwargs)


def autocast_ctx(enabled, device):
    if enabled and device.type == "cuda":
        return torch.cuda.amp.autocast()
    return nullcontext()

# =========================
# TRAIN
# =========================


def train_maskprob_gauss():
    set_seed(SEED)
    os.makedirs(OUT_DIR, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    msg = f"Device: {device} | UNetMaskProb base={BASE_CHANNELS} | LR={LR_INIT:g} | MAX_SIDE={MAX_SIDE} | loss={GAUSS_LOSS}"
    print(msg)
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        try:
            torch.set_float32_matmul_precision("high")
        except Exception:
            pass

    # data
    train_ds = MaskProbGaussDataset(TRAIN_IMG_DIR, TRAIN_MSK_DIR)
    val_ds = MaskProbGaussDataset(VAL_IMG_DIR,   VAL_MSK_DIR)
    train_ld = make_loader(train_ds, shuffle=True)
    val_ld = make_loader(val_ds,   shuffle=False)

    # model
    model = UNetMaskProb(in_ch=1, base=BASE_CHANNELS).to(device)
    if device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    if device.type == "cuda" and hasattr(torch, "compile") and USE_COMPILE:
        try:
            model = torch.compile(model, mode="max-autotune", dynamic=True)
            print("torch.compile: enabled")
        except Exception as e:
            print(f"torch.compile disabled at runtime ({e})")

    opt = torch.optim.AdamW(
        model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(
        enabled=(USE_AMP and device.type == "cuda"))
    ssim_h = SimpleSSIM(wt=GAUSS_SSIM_WEIGHT)

    best_val = float("inf")

    for epoch in range(1, EPOCHS+1):
        # ---- train
        model.train()
        tr_loss = 0.0
        for batch in train_ld:
            x = batch["image"].to(device, non_blocking=True)
            g = batch["gauss"].to(device, non_blocking=True)
            if device.type == "cuda":
                x = x.to(memory_format=torch.channels_last)
                g = g.to(memory_format=torch.channels_last)

            opt.zero_grad(set_to_none=True)
            with autocast_ctx(USE_AMP, device):
                p = model(x)
                l_gauss = gaussian_loss(p, g)
                l_ssim = ssim_h(p, g)
                loss = (1.0 - GAUSS_SSIM_WEIGHT) * \
                    l_gauss + GAUSS_SSIM_WEIGHT * l_ssim

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tr_loss += float(loss.item()) * x.size(0)

        tr_loss /= len(train_ds)

        # ---- val
        model.eval()
        va_loss = 0.0
        with torch.no_grad():
            for i, batch in enumerate(val_ld):
                x = batch["image"].to(device, non_blocking=True)
                g = batch["gauss"].to(device, non_blocking=True)
                if device.type == "cuda":
                    x = x.to(memory_format=torch.channels_last)
                    g = g.to(memory_format=torch.channels_last)
                with autocast_ctx(USE_AMP, device):
                    p = model(x)
                    l_gauss = gaussian_loss(p, g)
                    l_ssim = ssim_h(p, g)
                    loss = (1.0 - GAUSS_SSIM_WEIGHT) * \
                        l_gauss + GAUSS_SSIM_WEIGHT * l_ssim
                va_loss += float(loss.item()) * x.size(0)

                if i == 0 and (epoch % PREVIEW_EVERY == 0 or epoch == 1):
                    _save_preview(epoch, x[0, 0].float().cpu(
                    ), g[0, 0].float().cpu(), p[0, 0].float().cpu())

        va_loss /= len(val_ds)

        print(
            f"Epoch {epoch:03d} | train loss={tr_loss:.6f} | val loss={va_loss:.6f}")
        _append_csv(LOG_CSV, {"epoch": epoch, "lr": LR_INIT,
                    "train_loss": tr_loss, "val_loss": va_loss})

        if va_loss < best_val:
            best_val = va_loss
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_loss": va_loss,
                        "arch": "unet_maskprob_gauss", "base": BASE_CHANNELS}, CKPT_BEST)
            print(f"  ✓ saved {CKPT_BEST}")

    torch.save(model.state_dict(), CKPT_LAST)
    print(f"Saved last: {CKPT_LAST}\nCSV: {LOG_CSV}\nPreviews: {PREVIEW_DIR}")

# =========================
# (Optional) quick inference on VAL with best ckpt
# =========================


def run_infer_with_best():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ds = MaskProbGaussDataset(VAL_IMG_DIR, VAL_MSK_DIR)
    ld = make_loader(ds, shuffle=False)
    model = UNetMaskProb(in_ch=1, base=BASE_CHANNELS).to(device)
    state = torch.load(CKPT_BEST, map_location=device)
    sd = state["model"] if isinstance(
        state, dict) and "model" in state else state
    model.load_state_dict(sd, strict=True)
    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(ld):
            x = batch["image"].to(device, non_blocking=True)
            g = batch["gauss"].to(device, non_blocking=True)
            p = model(x)
            _save_preview(999, x[0, 0].float().cpu(),
                          g[0, 0].float().cpu(), p[0, 0].float().cpu())
            if i >= 9:
                break
    print("Inference previews saved to:", PREVIEW_DIR)


# =========================
# ENTRY
# =========================
if __name__ == "__main__":
    set_seed(SEED)
    if RUN_TRAIN:
        train_maskprob_gauss()
    if RUN_INFER:
        run_infer_with_best()

Device: cuda | UNetMaskProb base=32 | LR=0.0001 | MAX_SIDE=768 | loss=mse
[pairs] 9 from /kaggle/input/udids2025/UDIADS/images/train ↔ /kaggle/input/udids2025/UDIADS/masks/train
[pairs] 30 from /kaggle/input/udids2025/UDIADS/images/val ↔ /kaggle/input/udids2025/UDIADS/masks/val


/tmp/ipykernel_36/2941640520.py:496: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(


Epoch 001 | train loss=0.150953 | val loss=0.119338
  ✓ saved /kaggle/working/best_maskprob_gauss.pt
Epoch 002 | train loss=0.117401 | val loss=0.100501
  ✓ saved /kaggle/working/best_maskprob_gauss.pt
Epoch 003 | train loss=0.101049 | val loss=0.087939
  ✓ saved /kaggle/working/best_maskprob_gauss.pt
Epoch 004 | train loss=0.090110 | val loss=0.079384
  ✓ saved /kaggle/working/best_maskprob_gauss.pt
Epoch 005 | train loss=0.082370 | val loss=0.072945
  ✓ saved /kaggle/working/best_maskprob_gauss.pt
Epoch 006 | train loss=0.077329 | val loss=0.069528
  ✓ saved /kaggle/working/best_maskprob_gauss.pt
Epoch 007 | train loss=0.072616 | val loss=0.065569
  ✓ saved /kaggle/working/best_maskprob_gauss.pt
Epoch 008 | train loss=0.068917 | val loss=0.062530
  ✓ saved /kaggle/working/best_maskprob_gauss.pt
Epoch 009 | train loss=0.066444 | val loss=0.060725
  ✓ saved /kaggle/working/best_maskprob_gauss.pt
Epoch 010 | train loss=0.063854 | val loss=0.058304
  ✓ saved /kaggle/working/best_maskprob

In [3]:
import os
import random
from glob import glob
import numpy as np
from PIL import Image
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================
# CONFIG
# =========================
# Dataset layout assumed:
# {DATASET_ROOT}/images/{split}  where split in ["train","val","test"]
DATASET_ROOT = "/kaggle/input/udids2025/UDIADS"
IMAGES_SUBDIR = "images"
SPLITS = ["train", "val", "test"]

# Where your model #1 checkpoint lives
OUT_DIR = "/kaggle/working"
# falls back to 'last' if missing
CKPT_PATH = os.path.join(OUT_DIR, "best_maskprob_gauss.pt")

# Where to write priors (model #2 will read from here)
PRIORS_BASE_DIR = os.path.join(OUT_DIR, "priors")

# Model #1 architecture params (must match training)
BASE_CHANNELS = 32
MAX_SIDE = 768  # model’s test-time resize (longer side)

# Dataloader
BATCH_SIZE = 1
NUM_WORKERS = 0
PIN_MEMORY = True

# What to save
SAVE_NPY = True             # {name}.npy with float32 prior [H0,W0] in [0,1]
SAVE_PNG = True             # {name}.png (grayscale prior) for quick sanity check
SAVE_OVERLAY = False        # also save heatmap overlay for visual debugging
# used only to show a binarized preview (if you want)
PRED_THR = 0.30

SEED = 42
ALLOWED_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# =========================
# Utils
# =========================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def load_gray(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))

def save_png(arr: np.ndarray, path: str):
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    Image.fromarray(arr).save(path)

def _resize_keep_aspect_img(img_np: np.ndarray, max_side: int = MAX_SIDE):
    H, W = img_np.shape
    s = max(H, W)
    if s <= max_side:
        return img_np, 1.0
    scale = max_side / float(s)
    newW = int(round(W * scale))
    newH = int(round(H * scale))
    img_res = np.array(Image.fromarray(img_np).resize((newW, newH), Image.BILINEAR))
    return img_res, scale

# =========================
# Dataset (image-only)
# =========================

class SplitDataset(Dataset):
    def __init__(self, img_dir: str):
        assert os.path.isdir(img_dir), f"Missing {img_dir}"
        self.img_paths = sorted(
            p for p in glob(os.path.join(img_dir, "*"))
            if os.path.splitext(p)[1].lower() in ALLOWED_EXTS
        )
        if not self.img_paths:
            raise RuntimeError(f"No images in {img_dir}")
        print(f"[export priors] {len(self.img_paths)} images from {img_dir}")

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        ipath = self.img_paths[idx]
        name = os.path.splitext(os.path.basename(ipath))[0]
        img0 = load_gray(ipath)  # H0xW0 uint8
        H0, W0 = img0.shape

        img_res, _ = _resize_keep_aspect_img(img0, max_side=MAX_SIDE)
        x01 = (img_res.astype(np.float32) / 255.0)
        x_t = torch.from_numpy(x01)[None, ...]  # 1xHxW

        return {
            "image": x_t,         # tensor [1,H,W]
            "orig_image": img0,   # np.uint8 [H0,W0]
            "orig_shape": (H0, W0),
            "name": name
        }

def pad_to_max(batch):
    max_h = max(b["image"].shape[-2] for b in batch)
    max_w = max(b["image"].shape[-1] for b in batch)
    imgs = []
    meta = []
    for b in batch:
        x = b["image"]
        _, h, w = x.shape
        ph, pw = max_h - h, max_w - w
        if ph or pw:
            x = F.pad(x, (0, pw, 0, ph), value=0.0)
        imgs.append(x)
        meta.append({k: v for k, v in b.items() if k != "image"})
    return {"image": torch.stack(imgs, 0), "meta": meta}

# =========================
# Model (must match training)
# =========================

def _gn_groups(C: int) -> int:
    for g in (32, 16, 8, 4, 2, 1):
        if C % g == 0:
            return g
    return 1

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        g = _gn_groups(out_ch)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(g, out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(g, out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x): return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.reduce = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = self.reduce(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class UNetMaskProb(nn.Module):
    def __init__(self, in_ch=1, base=64):
        super().__init__()
        self.e1 = DoubleConv(in_ch, base)
        self.e2 = Down(base, base*2)
        self.e3 = Down(base*2, base*4)
        self.e4 = Down(base*4, base*8)
        self.bott = DoubleConv(base*8, base*16)
        self.u4 = Up(base*16, base*8, base*8)
        self.u3 = Up(base*8,  base*4, base*4)
        self.u2 = Up(base*4,  base*2, base*2)
        self.u1 = Up(base*2,  base,   base)
        self.head = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        s4 = self.e4(s3)
        z = self.bott(s4)
        z = self.u4(z, s4)
        z = self.u3(z, s3)
        z = self.u2(z, s2)
        z = self.u1(z, s1)
        return torch.sigmoid(self.head(z))  # [B,1,H,W] in [0,1]

# =========================
# Export (multi-split)
# =========================

def ensure_checkpoint_path():
    if not os.path.isfile(CKPT_PATH):
        alt = os.path.join(OUT_DIR, "last_maskprob_gauss.pt")
        if os.path.isfile(alt):
            print(f"[warn] {CKPT_PATH} not found, using {alt}")
            return alt
        raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")
    return CKPT_PATH

def export_split(split: str, model: nn.Module, device: torch.device) -> int:
    img_dir = os.path.join(DATASET_ROOT, IMAGES_SUBDIR, split)
    out_dir = os.path.join(PRIORS_BASE_DIR, split)

    if not os.path.isdir(img_dir):
        print(f"[skip] Split '{split}' missing: {img_dir}")
        return 0

    os.makedirs(out_dir, exist_ok=True)

    ds = SplitDataset(img_dir)
    ld = DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, collate_fn=pad_to_max, pin_memory=PIN_MEMORY
    )

    saved = 0
    model.eval()
    with torch.no_grad():
        for batch in ld:
            x = batch["image"].to(device, non_blocking=True)  # [B,1,h,w]
            if device.type == "cuda":
                x = x.to(memory_format=torch.channels_last)
            p = model(x).float().cpu().squeeze(1).numpy()      # [B,h,w] in [0,1]
            metas = batch["meta"]

            for b in range(p.shape[0]):
                name = metas[b]["name"]
                H0, W0 = metas[b]["orig_shape"]
                img0 = metas[b]["orig_image"]

                prob_res = p[b]
                prob_orig = cv2.resize(prob_res, (W0, H0), interpolation=cv2.INTER_LINEAR)

                # --- save prior (.npy) ---
                if SAVE_NPY:
                    np.save(os.path.join(out_dir, f"{name}.npy"), prob_orig.astype(np.float32))

                # Optional visualizations
                if SAVE_PNG:
                    prob_u8 = np.clip(prob_orig * 255.0, 0, 255).astype(np.uint8)
                    save_png(prob_u8, os.path.join(out_dir, f"{name}.png"))
                if SAVE_OVERLAY:
                    base3 = cv2.cvtColor(img0, cv2.COLOR_GRAY2BGR)
                    prob_cmap = cv2.applyColorMap(
                        np.clip(prob_orig*255, 0, 255).astype(np.uint8), cv2.COLORMAP_JET
                    )
                    overlay = cv2.addWeighted(base3, 0.4, prob_cmap, 0.6, 0)
                    cv2.imwrite(os.path.join(out_dir, f"{name}_overlay.png"), overlay)
                    mask_bin = (prob_orig > PRED_THR).astype(np.uint8) * 255
                    save_png(mask_bin, os.path.join(out_dir, f"{name}_mask_thr{int(PRED_THR*100)}.png"))
                saved += 1

    print(f"[done] Split '{split}': saved {saved} priors -> {out_dir}")
    return saved

def main():
    set_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    # Prepare model once
    model = UNetMaskProb(in_ch=1, base=BASE_CHANNELS).to(device)
    if device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    ckpt_to_load = ensure_checkpoint_path()
    state = torch.load(ckpt_to_load, map_location=device)
    sd = state["model"] if isinstance(state, dict) and "model" in state else state
    model.load_state_dict(sd, strict=True)

    # Process all requested splits
    total_saved = 0
    for split in SPLITS:
        total_saved += export_split(split, model, device)

    print(f"All splits completed. Total priors saved: {total_saved}")
    print(f"Priors base dir: {PRIORS_BASE_DIR}")

if __name__ == "__main__":
    main()


Device: cuda
[export priors] 9 images from /kaggle/input/udids2025/UDIADS/images/train
[done] Split 'train': saved 9 priors -> /kaggle/working/priors/train
[export priors] 30 images from /kaggle/input/udids2025/UDIADS/images/val
[done] Split 'val': saved 30 priors -> /kaggle/working/priors/val
[export priors] 43 images from /kaggle/input/udids2025/UDIADS/images/test
[done] Split 'test': saved 43 priors -> /kaggle/working/priors/test
All splits completed. Total priors saved: 82
Priors base dir: /kaggle/working/priors


# Stage 2

In [4]:
import os
import random
from glob import glob
from typing import Tuple, Dict, Optional, List
import numpy as np
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext
try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# =========================
# CONFIG — EDIT THESE
# =========================
IMG_TRAIN = "/kaggle/input/udids2025/UDIADS/images/train"
IMG_VAL = "/kaggle/input/udids2025/UDIADS/images/val"
TL_MSK_TRAIN = "/kaggle/input/udids2025/UDIADS/masks/train"
TL_MSK_VAL = "/kaggle/input/udids2025/UDIADS/masks/val"
PRIOR_DIR_TRAIN = "/kaggle/working/priors/train"
PRIOR_DIR_VAL = "/kaggle/working/priors/val"

OUT_DIR = "/kaggle/working/Textlines"
CKPT_BEST = os.path.join(OUT_DIR, "best_lineiu.pt")
CKPT_LAST = os.path.join(OUT_DIR, "last_lineiu.pt")
LOG_CSV = os.path.join(OUT_DIR, "metrics_lineiu.csv")
PREV_DIR = os.path.join(OUT_DIR, "preview")

EPOCHS = 500
BATCH_SIZE = 1
BASE = 64
LR_INIT = 1e-4
WEIGHT_DECAY = 0.0
AMP = torch.cuda.is_available()
GRAD_CLIP = 1.0

MAX_SIDE = 2048
PATCH = 512
TRAIN_PATCHES_PER_IMAGE = 8
POS_PATCH_PROB = 0.6
VAL_STRIDE = PATCH // 2

NUM_WORKERS = 0
PIN_MEMORY = True

CASCADE_USE_PRIOR_AS_INPUT = True
TEACHER_LOSS_W = 0.0
TEACHER_LOSS = "mse"

# --- Prior fusion knobs ---
ADAPTIVE_ALPHA = True          # <— enable per-pixel α-map (recommended)
ALPHA_MIN = 0.15               # lower bound for α-map
# upper bound for α-map  (much softer than fixed 3.0)
ALPHA_MAX = 1.50
ALPHA_FIXED = 3.0              # kept for compatibility; ignored when ADAPTIVE_ALPHA=True
BETA_GATE = 1.5              # soften residual gating (1.0–2.0 good)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ALLOWED = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# =========================
# I/O
# =========================


def load_rgb(p): return np.array(Image.open(p).convert("RGB"))


def load_mask01(p): return (
    np.array(Image.open(p).convert("L")) > 0).astype(np.uint8)


def resize_keep(img, max_side=MAX_SIDE):
    H, W = img.shape[:2]
    s = max(H, W)
    if s <= max_side:
        return img, 1.0
    sc = max_side/float(s)
    interp = cv2.INTER_AREA if sc < 1.0 and img.ndim == 3 else cv2.INTER_LINEAR
    out = cv2.resize(img, (int(round(W*sc)), int(round(H*sc))),
                     interpolation=interp)
    return out, sc

# =========================
# Dataset
# =========================


class TLDS(Dataset):
    def __init__(self, img_dir, tl_mask_dir, prior_dir=None, use_prior=True):
        imgs = sorted(p for p in glob(os.path.join(img_dir, "*"))
                      if os.path.splitext(p)[1].lower() in ALLOWED)
        tmsk = sorted(p for p in glob(os.path.join(tl_mask_dir, "*"))
                      if os.path.splitext(p)[1].lower() in ALLOWED)
        imap = {os.path.splitext(os.path.basename(p))[0]: p for p in imgs}
        mmap = {os.path.splitext(os.path.basename(p))[0]: p for p in tmsk}
        self.items = []
        for k, ip in imap.items():
            if k in mmap:
                self.items.append((ip, mmap[k], k))
        if not self.items:
            raise RuntimeError(
                f"No image/textline pairs in {img_dir} ↔ {tl_mask_dir}")
        self.prior_dir, self.use_prior = prior_dir, use_prior
        if self.use_prior and not self.prior_dir:
            raise RuntimeError(
                "CASCADE_USE_PRIOR_AS_INPUT=True but PRIOR_DIR is None")
        print(f"[pairs] {len(self.items)} (prior_dir={self.prior_dir})")

    def _load_prior(self, key, H0, W0):
        if not self.use_prior:
            return np.zeros((H0, W0), np.float32)
        npy = os.path.join(self.prior_dir, f"{key}.npy")
        png = os.path.join(self.prior_dir, f"{key}.png")
        if os.path.isfile(npy):
            p = np.load(npy).astype(np.float32)
            p = p/255.0 if p.max() > 1.0 else p
        elif os.path.isfile(png):
            p = np.array(Image.open(png).convert("F"))/255.0
        else:
            raise FileNotFoundError(
                f"Missing prior for '{key}' in {self.prior_dir}")
        if p.shape != (H0, W0):
            p = cv2.resize(p, (W0, H0), interpolation=cv2.INTER_LINEAR)
        return np.clip(p, 0, 1)

    def __len__(self): return len(self.items)

    def __getitem__(self, i) -> Dict[str, torch.Tensor]:
        ipath, mpath, key = self.items[i]
        img0 = load_rgb(ipath)
        tl0 = load_mask01(mpath)
        H0, W0 = tl0.shape
        prior0 = self._load_prior(
            key, H0, W0) if self.use_prior else np.zeros_like(tl0, np.float32)

        img, _ = resize_keep(img0, MAX_SIDE)
        tl = cv2.resize(
            tl0,   img.shape[:2][::-1], interpolation=cv2.INTER_NEAREST).astype(np.float32)
        prior = cv2.resize(
            prior0, img.shape[:2][::-1], interpolation=cv2.INTER_LINEAR).astype(np.float32)

        x_rgb = (img.astype(np.float32)/255.0).transpose(2, 0, 1)
        x = np.concatenate([x_rgb, prior[None, ...]],
                           0) if self.use_prior else x_rgb

        return {"x": torch.from_numpy(x),
                "y": torch.from_numpy(tl[None, ...]),
                "prior": torch.from_numpy(np.clip(prior, 0, 1)[None, ...]),
                "rgb": torch.from_numpy(x_rgb),
                "name": os.path.basename(ipath)}

# =========================
# Model — UNet with ADAPTIVE α-map + prior-gated residuals
# =========================


def _gn(C):
    for g in (32, 16, 8, 4, 2, 1):
        if C % g == 0:
            return g
    return 1


class Block(nn.Module):
    def __init__(self, c_in, c_out, p=0.0):
        super().__init__()
        g = _gn(c_out)
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.GroupNorm(
                g, c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.GroupNorm(
                g, c_out), nn.ReLU(inplace=True),
            nn.Dropout2d(p) if p > 0 else nn.Identity())

    def forward(self, x): return self.net(x)


class Down(nn.Module):
    def __init__(self, c_in, c_out, p=0.0):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.block = Block(c_in, c_out, p)

    def forward(self, x): return self.block(self.pool(x))


class Up(nn.Module):
    def __init__(self, c_in, c_skip, c_out, p=0.0):
        super().__init__()
        self.red = nn.Conv2d(c_in, c_out, 1, bias=False)
        self.block = Block(c_out+c_skip, c_out, p)

    def forward(self, x, s):
        x = F.interpolate(x, size=s.shape[-2:],
                          mode="bilinear", align_corners=False)
        x = self.red(x)
        x = torch.cat([x, s], 1)
        return self.block(x)


def _safe_logit(p, eps=1e-6): p = p.clamp(eps, 1 -
                                          eps); return torch.log(p)-torch.log(1-p)


class UNetTextLines(nn.Module):
    """
    Fused logits = α_map · prior_logits + gate · (prior^beta · RES_SCALE · tanh(delta))
    α_map in [ALPHA_MIN, ALPHA_MAX] is predicted per-pixel (so prior influence is reduced where it’s wrong).
    """

    def __init__(self, in_ch=4, base=64, p=0.1,
                 alpha_min=ALPHA_MIN, alpha_max=ALPHA_MAX, beta_gate=BETA_GATE, res_scale=2.0):
        super().__init__()
        self.in_ch = in_ch
        self.alpha_min = float(alpha_min)
        self.alpha_max = float(alpha_max)
        self.beta_gate = float(beta_gate)
        self.res_scale = float(res_scale)

        self.e1 = Block(in_ch, base, p)
        self.e2 = Down(base, base*2, p)
        self.e3 = Down(base*2, base*4, p)
        self.e4 = Down(base*4, base*8, p)
        self.bott = Block(base*8, base*16, p)
        self.u4 = Up(base*16, base*8, base*8, p)
        self.u3 = Up(base*8, base*4, base*4, p)
        self.u2 = Up(base*4, base*2, base*2, p)
        self.u1 = Up(base*2, base, base, 0.0)

        self.head_delta = nn.Conv2d(base, 2, 1)    # residual 2-class logits
        self.head_gate = nn.Conv2d(base, 1, 1)    # residual gate in (0,1)
        # per-pixel α-map in [αmin, αmax]
        self.head_alpha = nn.Conv2d(base, 1, 1)

        nn.init.zeros_(self.head_delta.weight)
        nn.init.zeros_(self.head_delta.bias)
        nn.init.zeros_(self.head_gate.weight)
        nn.init.constant_(self.head_gate.bias, -2.0)     # small residuals
        nn.init.zeros_(self.head_alpha.weight)
        nn.init.constant_(self.head_alpha.bias, 0.0)     # start midrange

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        s4 = self.e4(s3)
        z = self.bott(s4)
        z = self.u4(z, s4)
        z = self.u3(z, s3)
        z = self.u2(z, s2)
        z = self.u1(z, s1)

        if self.in_ch >= 4:
            prior = x[:, 3:4].clamp(0, 1)
            l_fg = _safe_logit(prior)
            l_bg = _safe_logit(1.0-prior)
            prior_logits = torch.cat([l_bg, l_fg], 1)   # [B,2,H,W]
            mask = prior.pow(self.beta_gate)
        else:
            B, _, H, W = x.shape
            prior_logits = torch.zeros(
                B, 2, H, W, device=x.device, dtype=x.dtype)
            mask = 0.0

        # residual inside bands
        delta = self.res_scale*torch.tanh(self.head_delta(z))
        if isinstance(mask, torch.Tensor):
            delta = mask*delta
        gate = torch.sigmoid(self.head_gate(z))

        # adaptive α-map ∈ [αmin, αmax]
        a = torch.sigmoid(self.head_alpha(z))
        alpha_map = self.alpha_min + \
            (self.alpha_max - self.alpha_min)*a      # [B,1,H,W]
        fused_logits = alpha_map*prior_logits + gate * \
            delta                    # broadcast α over 2 classes
        return fused_logits, alpha_map

# =========================
# Metrics / helpers
# =========================


def evaluate_metrics_np(gt_u8, pr_u8, thresh=0.75):
    ng, gt_lbl = cv2.connectedComponents(gt_u8)
    np_, pr_lbl = cv2.connectedComponents(pr_u8)
    inter = np.logical_and(gt_lbl > 0, pr_lbl > 0).sum()
    union = np.logical_or(gt_lbl > 0, pr_lbl > 0).sum()
    pixel_IU = inter/union if union > 0 else 0.0
    M, N = ng, np_
    if M <= 1 or N <= 1:
        return pixel_IU, 0.0, 0.0, 0.0, 0.0
    table = np.bincount(gt_lbl.ravel()*N+pr_lbl.ravel(),
                        minlength=M*N).reshape(M, N)
    area_gt = table.sum(1)[:, None]
    area_pr = table.sum(0)[None, :]
    iou = table/(area_gt+area_pr-table+1e-8)
    sub = iou[1:, 1:]
    best_pr = sub.argmax(1)+1
    gt_i = np.arange(1, M)
    pr_i = best_pr
    ints = table[gt_i, pr_i]
    precs = ints/(area_pr[0, pr_i]+1e-8)
    recs = ints/(area_gt[gt_i, 0]+1e-8)
    CL = (precs >= thresh) & (recs >= thresh)
    ML = (recs < thresh)
    EL = (precs < thresh) & (recs >= thresh)
    line_IU = CL.sum()/max(1, (CL.sum()+ML.sum()+EL.sum()))
    rows, cols = np.where((iou >= thresh) & (
        np.arange(M)[:, None] > 0) & (np.arange(N)[None, :] > 0))
    matches = len(rows)
    DR = matches/max(1, (M-1))
    RA = matches/max(1, (N-1))
    FM = 2*DR*RA/(DR+RA+1e-8)
    return pixel_IU, line_IU, DR, RA, FM


def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


def sample_random_patch(H, W, ph, pw, yc=None, xc=None):
    if yc is not None and xc is not None:
        return max(0, min(H-ph, int(yc-ph//2))), max(0, min(W-pw, int(xc-pw//2)))
    y0 = 0 if H <= ph else random.randint(0, H-ph)
    x0 = 0 if W <= pw else random.randint(0, W-pw)
    return y0, x0


def cosine_window_2d(h, w, eps=1e-3):
    wy = np.hanning(h) if h > 1 else np.ones(1)
    wx = np.hanning(w) if w > 1 else np.ones(1)
    w2 = np.outer(wy, wx).astype(np.float32)
    return np.clip(w2, eps, 1.0)


@torch.no_grad()
def tiled_predict_prob(model, x_full, device, patch=PATCH, stride=VAL_STRIDE):
    C, H, W = x_full.shape
    acc = torch.zeros((1, H, W), dtype=torch.float32, device=device)
    wgt = torch.zeros((1, H, W), dtype=torch.float32, device=device)
    win_cache: Dict[Tuple[int, int], torch.Tensor] = {}
    model.eval()
    ys = list(range(0, max(1, H-patch+1), stride))
    xs = list(range(0, max(1, W-patch+1), stride))
    edge_y = max(0, H-patch)
    edge_x = max(0, W-patch)
    if ys[-1] != edge_y:
        ys.append(edge_y)
    if xs[-1] != edge_x:
        xs.append(edge_x)
    it = [(y, x) for y in ys for x in xs]
    it = tqdm(it, desc="Val tiling", leave=False) if tqdm else it
    for (y0, x0) in it:
        y1 = min(H, y0+patch)
        x1 = min(W, x0+patch)
        tile = x_full[:, y0:y1, x0:x1].unsqueeze(0).to(device)
        fused_logits, _ = model(tile)                 # [1,2,h,w] , [1,1,h,w]
        p = torch.softmax(fused_logits, dim=1)[:, 1:2]   # [1,1,h,w]
        h, w = y1-y0, x1-x0
        key = (h, w)
        if key not in win_cache:
            win_cache[key] = torch.from_numpy(cosine_window_2d(
                h, w)).to(device=device, dtype=torch.float32)
        win_t = win_cache[key]
        acc[:, y0:y1, x0:x1] += p.squeeze(0)*win_t
        wgt[:, y0:y1, x0:x1] += win_t
    return torch.where(wgt > 0, acc/wgt, acc).clamp(0, 1)

# =========================
# Train
# =========================


def train_lineiu():
    os.makedirs(OUT_DIR, exist_ok=True)
    os.makedirs(PREV_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        try:
            torch.set_float32_matmul_precision("high")
        except:
            pass

    ds_tr = TLDS(IMG_TRAIN, TL_MSK_TRAIN, PRIOR_DIR_TRAIN,
                 CASCADE_USE_PRIOR_AS_INPUT)
    ds_va = TLDS(IMG_VAL,  TL_MSK_VAL,  PRIOR_DIR_VAL,
                 CASCADE_USE_PRIOR_AS_INPUT)
    ld_tr = make_loader(ds_tr, True)
    ld_va = make_loader(ds_va, False)

    in_ch = (3+1) if CASCADE_USE_PRIOR_AS_INPUT else 3
    model = UNetTextLines(in_ch=in_ch, base=BASE, p=0.1,
                          alpha_min=ALPHA_MIN, alpha_max=ALPHA_MAX, beta_gate=BETA_GATE).to(device)
    if device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    opt = torch.optim.AdamW(
        model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=EPOCHS, eta_min=LR_INIT*0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    ce_loss = nn.CrossEntropyLoss()

    best_lineiu = -1.0

    for ep in range(1, EPOCHS+1):
        # ---- train
        model.train()
        trL = 0.0
        itr = tqdm(ld_tr, desc=f"Train {ep}/{EPOCHS}") if tqdm else ld_tr
        for b in itr:
            x_full = b["x"].to(device, non_blocking=True).float()
            y_full = b["y"].to(device, non_blocking=True).float()
            if device.type == "cuda":
                x_full = x_full.to(memory_format=torch.channels_last)
                y_full = y_full.to(memory_format=torch.channels_last)

            _, _, H, W = x_full.shape
            for _ in range(TRAIN_PATCHES_PER_IMAGE):
                yc = xc = None
                if random.random() < POS_PATCH_PROB:
                    pos = torch.nonzero(y_full[0, 0] > 0.5, as_tuple=False)
                    if pos.numel() > 0:
                        yidx = pos[random.randrange(pos.shape[0])].tolist()
                        yc, xc = yidx[0], yidx[1]
                y0, x0 = sample_random_patch(H, W, PATCH, PATCH, yc, xc)
                y1 = min(H, y0+PATCH)
                x1 = min(W, x0+PATCH)
                x = x_full[:, :, y0:y1, x0:x1]
                y = y_full[:, :, y0:y1, x0:x1]
                target = (y > 0.5).long().squeeze(1)

                opt.zero_grad(set_to_none=True)
                with (torch.cuda.amp.autocast() if AMP else nullcontext()):
                    fused_logits, _ = model(x)
                    loss = ce_loss(fused_logits, target)
                scaler.scale(loss).backward()
                if GRAD_CLIP:
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt)
                scaler.update()
                trL += float(loss.item())*x.size(0)
        trL /= (len(ds_tr)*max(1, TRAIN_PATCHES_PER_IMAGE))

        # ---- validation
        model.eval()
        vaL = 0.0
        dices = []
        lineius = []
        alpha_means = []

        best_preview_rgb = best_preview_prob = best_preview_gt = best_preview_prior = None

        itr = tqdm(ld_va, desc=f"Val {ep}/{EPOCHS}") if tqdm else ld_va
        with torch.no_grad():
            for i, b in enumerate(itr):
                x_full = b["x"][0].float().to(device)
                y_full = b["y"][0].float().to(device)
                target_full = (y_full > 0.5).long().squeeze(0)

                # get prob + alpha map from full tiling
                C, H, W = x_full.shape
                acc_alpha = torch.zeros((1, H, W), device=device)
                wgt_alpha = torch.zeros((1, H, W), device=device)

                # reuse tiler here to also average alpha
                ys = list(range(0, max(1, H-PATCH+1), VAL_STRIDE))
                xs = list(range(0, max(1, W-PATCH+1), VAL_STRIDE))
                ey = max(0, H-PATCH)
                ex = max(0, W-PATCH)
                if ys[-1] != ey:
                    ys.append(ey)
                if xs[-1] != ex:
                    xs.append(ex)
                acc = torch.zeros((1, H, W), device=device)
                wgt = torch.zeros((1, H, W), device=device)
                win_cache = {}
                for (y0, x0) in [(yy, xx) for yy in ys for xx in xs]:
                    y1 = min(H, y0+PATCH)
                    x1 = min(W, x0+PATCH)
                    tile = x_full[:, y0:y1, x0:x1].unsqueeze(0).to(device)
                    # [1,2,h,w], [1,1,h,w]
                    fused_logits, a_map = model(tile)
                    p = torch.softmax(fused_logits, dim=1)[:, 1:2]  # [1,1,h,w]
                    h, w = y1-y0, x1-x0
                    key = (h, w)
                    if key not in win_cache:
                        win_cache[key] = torch.from_numpy(cosine_window_2d(
                            h, w)).to(device=device, dtype=torch.float32)
                    win = win_cache[key]
                    acc[:, y0:y1, x0:x1] += p.squeeze(0)*win
                    wgt[:, y0:y1, x0:x1] += win
                    acc_alpha[:, y0:y1, x0:x1] += a_map.squeeze(0)*win
                    wgt_alpha[:, y0:y1, x0:x1] += win
                p_prob = torch.where(wgt > 0, acc/wgt, acc).clamp(0, 1)
                a_full = torch.where(
                    wgt_alpha > 0, acc_alpha/wgt_alpha, acc_alpha)
                alpha_means.append(float(a_full.mean().cpu()))

                # reporting CE (rebuild logits from prob)
                p = p_prob.clamp(1e-6, 1-1e-6)
                l_fg = torch.log(p)-torch.log(1-p)
                l_bg = -l_fg
                logits_full = torch.cat([l_bg, l_fg], 0).unsqueeze(0)
                loss_val = ce_loss(logits_full, target_full.unsqueeze(0))
                vaL += float(loss_val.item())

                # metrics
                pp = (p_prob >= 0.5).float()
                gg = (y_full > 0.5).float()
                tp = (pp*gg).sum().item()
                fp = (pp*(1-gg)).sum().item()
                fn = ((1-pp)*gg).sum().item()
                dices.append((2*tp)/(2*tp+fp+fn+1e-6))
                gt_u8 = (gg[0].cpu().numpy().astype(np.uint8))*255
                pr_u8 = (pp[0].cpu().numpy().astype(np.uint8))*255
                _, liu, _, _, _ = evaluate_metrics_np(
                    gt_u8, pr_u8, thresh=0.75)
                lineius.append(liu)

                if i == 0:
                    best_preview_rgb = (b["rgb"][0].numpy().transpose(
                        1, 2, 0)*255).astype(np.uint8)
                    best_preview_prob = p_prob[0].cpu(
                    ).numpy().astype(np.float32)
                    best_preview_gt = (
                        y_full[0].cpu().numpy() > 0.5).astype(np.uint8)
                    if "prior" in b:
                        best_preview_prior = (b["prior"][0, 0].cpu(
                        ).numpy().clip(0, 1)*255).astype(np.uint8)

                if i == 0 and (ep == 1 or ep % 25 == 0):
                    rgb = (b["rgb"][0].numpy().transpose(
                        1, 2, 0)*255).astype(np.uint8)
                    gp = (p_prob[0].cpu().numpy()*255).astype(np.uint8)
                    gt = (gg[0].cpu().numpy()*255).astype(np.uint8)
                    rgb_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
                    ovp = cv2.addWeighted(rgb_bgr, 0.5, cv2.applyColorMap(
                        gp, cv2.COLORMAP_JET), 0.6, 0)
                    ovg = cv2.addWeighted(rgb_bgr, 0.5, cv2.applyColorMap(
                        gt, cv2.COLORMAP_JET), 0.6, 0)
                    os.makedirs(PREV_DIR, exist_ok=True)
                    cv2.imwrite(os.path.join(
                        PREV_DIR, f"ep{ep:03d}_img.png"), rgb_bgr)
                    cv2.imwrite(os.path.join(
                        PREV_DIR, f"ep{ep:03d}_pred.png"), gp)
                    cv2.imwrite(os.path.join(
                        PREV_DIR, f"ep{ep:03d}_gt.png"), gt)
                    cv2.imwrite(os.path.join(
                        PREV_DIR, f"ep{ep:03d}_ov_pred.png"), ovp)
                    cv2.imwrite(os.path.join(
                        PREV_DIR, f"ep{ep:03d}_ov_gt.png"), ovg)
                    if "prior" in b:
                        pr8 = (b["prior"][0, 0].cpu().numpy().clip(
                            0, 1)*255).astype(np.uint8)
                        cv2.imwrite(os.path.join(
                            PREV_DIR, f"ep{ep:03d}_gauss.png"), pr8)

        vaL /= max(1, len(ds_va))
        vaDice = float(np.mean(dices)) if dices else 0.0
        vaLineIU = float(np.mean(lineius)) if lineius else 0.0
        alpha_mean = float(np.mean(alpha_means)) if alpha_means else 0.0
        sched.step()

        print(f"Epoch {ep:03d} | trainCE {trL:.5f} | valCE {vaL:.5f} | valDice {vaDice:.4f} | "
              f"valLineIU {vaLineIU:.4f} | α_mean {alpha_mean:.3f} "
              f"[α∈{ALPHA_MIN:.2f},{ALPHA_MAX:.2f}] beta {BETA_GATE}")

        # log
        os.makedirs(os.path.dirname(LOG_CSV), exist_ok=True)
        import csv
        hdr = not os.path.isfile(LOG_CSV)
        with open(LOG_CSV, "a", newline="") as f:
            w = csv.writer(f)
            if hdr:
                w.writerow(["epoch", "lr", "train_ce", "val_ce", "val_dice",
                           "val_lineiu", "alpha_mean", "alpha_min", "alpha_max", "beta_gate"])
            w.writerow([ep, opt.param_groups[0]["lr"], trL, vaL, vaDice,
                       vaLineIU, alpha_mean, ALPHA_MIN, ALPHA_MAX, BETA_GATE])

        # save by best LineIU
        state = {"model": model.state_dict(), "epoch": ep, "val_ce": vaL, "val_dice": vaDice, "val_lineiu": vaLineIU,
                 "arch": "unet_textlines_lineiu_rgb", "base": BASE, "in_ch": in_ch,
                 "cascade_prior_input": CASCADE_USE_PRIOR_AS_INPUT,
                 "patch": PATCH, "val_stride": VAL_STRIDE,
                 "alpha_min": ALPHA_MIN, "alpha_max": ALPHA_MAX, "beta_gate": BETA_GATE}
        torch.save(state, CKPT_LAST)
        if vaLineIU > best_lineiu:
            best_lineiu = vaLineIU
            torch.save(state, CKPT_BEST)
            print(f"  ✓ saved {CKPT_BEST} (LineIU={vaLineIU:.4f})")
            if best_preview_rgb is not None:
                os.makedirs(PREV_DIR, exist_ok=True)
                rgb_bgr = cv2.cvtColor(best_preview_rgb, cv2.COLOR_RGB2BGR)
                gp = (best_preview_prob*255.0).astype(np.uint8)
                gt = (best_preview_gt*255).astype(np.uint8)
                ovp = cv2.addWeighted(rgb_bgr, 0.5, cv2.applyColorMap(
                    gp, cv2.COLORMAP_JET), 0.6, 0)
                ovg = cv2.addWeighted(rgb_bgr, 0.5, cv2.applyColorMap(
                    gt, cv2.COLORMAP_JET), 0.6, 0)
                cv2.imwrite(os.path.join(
                    PREV_DIR, f"ep{ep:03d}_best_img.png"), rgb_bgr)
                cv2.imwrite(os.path.join(
                    PREV_DIR, f"ep{ep:03d}_best_pred.png"), gp)
                cv2.imwrite(os.path.join(
                    PREV_DIR, f"ep{ep:03d}_best_ov_pred.png"), ovp)
                cv2.imwrite(os.path.join(
                    PREV_DIR, f"ep{ep:03d}_best_ov_gt.png"), ovg)
                if best_preview_prior is not None:
                    cv2.imwrite(os.path.join(
                        PREV_DIR, f"ep{ep:03d}_best_gauss.png"), best_preview_prior)

    print("Done. Best LineIU:", best_lineiu)


if __name__ == "__main__":
    train_lineiu()

[pairs] 9 (prior_dir=/kaggle/working/priors/train)
[pairs] 30 (prior_dir=/kaggle/working/priors/val)


/tmp/ipykernel_36/3181531310.py:415: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=AMP)
Train 1/500:   0%|          | 0/9 [00:00<?, ?it/s]/tmp/ipykernel_36/3181531310.py:448: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with (torch.cuda.amp.autocast() if AMP else nullcontext()):
Val 1/500: 100%|██████████| 30/30 [01:38<00:00,  3.28s/it]


Epoch 001 | trainCE 0.31469 | valCE 0.21344 | valDice 0.7623 | valLineIU 0.4806 | α_mean 0.812 [α∈0.15,1.50] beta 1.5
  ✓ saved /kaggle/working/Textlines/best_lineiu.pt (LineIU=0.4806)


Val 2/500: 100%|██████████| 30/30 [01:36<00:00,  3.21s/it]


Epoch 002 | trainCE 0.25208 | valCE 0.18686 | valDice 0.8193 | valLineIU 0.7410 | α_mean 0.802 [α∈0.15,1.50] beta 1.5
  ✓ saved /kaggle/working/Textlines/best_lineiu.pt (LineIU=0.7410)


...

Epoch 274 | trainCE 0.08951 | valCE 0.10273 | valDice 0.9104 | valLineIU 0.9322 | α_mean 1.343 [α∈0.15,1.50] beta 1.5


Val 275/500: 100%|██████████| 30/30 [01:36<00:00,  3.22s/it]


Epoch 275 | trainCE 0.08446 | valCE 0.10446 | valDice 0.9105 | valLineIU 0.9272 | α_mean 1.344 [α∈0.15,1.50] beta 1.5


Val 276/500:  87%|████████▋ | 26/30 [01:26<00:13,  3.34s/it]


KeyboardInterrupt: 

In [7]:
# test_textlines_lineiu_rgb_patches.py
# Inference for the adaptive-α fusion model (CE-only, softmax over fused logits).

import os
import random
from glob import glob
from typing import Tuple, Dict, Optional
import numpy as np
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

# =========================
# CONFIG
# =========================
TEST_IMG_DIR = "/kaggle/input/udids2025/UDIADS/images/test"               # or "Latin2/images/test"
# set to None to skip metrics
TL_MSK_TEST = "/kaggle/input/udids2025/UDIADS/masks/test"
PRIOR_DIR_TEST = "/kaggle/working//priors/test"     # or ".../priors/test"

CKPT_PATH = "/kaggle/working/Textlines/best_lineiu.pt"

PRED_DIR = "/kaggle/working/Textlines/preds"
SAVE_PROB = True
SAVE_OVERLAY = True
SAVE_NPY = False
SAVE_PRIOR = True
SAVE_PRIOR_OVERLAY = True
SAVE_ALPHA = False        # NEW: save averaged α-map and overlay for debugging

PRED_THR = 0.5
MAX_SIDE = 2048

DEFAULT_BASE = 64
DEFAULT_PATCH = 512
DEFAULT_STRIDE = DEFAULT_PATCH // 2

BATCH_SIZE = 1
NUM_WORKERS = 0
PIN_MEMORY = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ALLOWED = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

# =========================
# I/O
# =========================


def load_rgb(p): return np.array(Image.open(p).convert("RGB"))


def load_mask01(p): return (
    np.array(Image.open(p).convert("L")) > 0).astype(np.uint8)


def resize_keep(img, max_side=MAX_SIDE):
    H, W = img.shape[:2]
    s = max(H, W)
    if s <= max_side:
        return img, 1.0
    sc = max_side/float(s)
    inter = cv2.INTER_AREA if sc < 1.0 and img.ndim == 3 else cv2.INTER_LINEAR
    out = cv2.resize(img, (int(round(W*sc)), int(round(H*sc))),
                     interpolation=inter)
    return out, sc

# =========================
# Dataset (reads priors; no fallback)
# =========================


class TestDS(Dataset):
    """
    Yields resized full-image tensors and metadata:
      x: [C,H,W] in [0,1] (RGB or RGB+prior)
      rgb_resized: [H,W,3] uint8
      prior_resized: [H,W] float32 in [0,1]
      prior_orig: [H0,W0] float32 in [0,1] (if use_prior)
      orig_rgb: [H0,W0,3] uint8
      orig_shape: (H0, W0)
      res_shape:  (H, W)
      name: basename
    Optional GT:
      gt_orig: [H0,W0] uint8 {0,1}
      gt_resized: [H,W] uint8 {0,1}
    """

    def __init__(self, img_dir, prior_dir, use_prior, gt_dir: Optional[str] = None):
        assert os.path.isdir(img_dir), f"Missing {img_dir}"
        self.use_prior = use_prior
        self.prior_dir = prior_dir
        self.gt_dir = gt_dir
        if self.use_prior and not self.prior_dir:
            raise RuntimeError(
                "Model expects priors but PRIOR_DIR_TEST is None.")

        self.img_paths = sorted(
            p for p in glob(os.path.join(img_dir, "*"))
            if os.path.splitext(p)[1].lower() in ALLOWED
        )
        if not self.img_paths:
            raise RuntimeError(f"No images in {img_dir}")
        print(f"[test] {len(self.img_paths)} images from {img_dir}")
        if self.use_prior:
            print(f"[test] Using priors from: {self.prior_dir}")
        if self.gt_dir:
            print(f"[test] Will compute metrics using GT from: {self.gt_dir}")

    def _load_prior(self, key, H0, W0):
        if not self.use_prior:
            return np.zeros((H0, W0), np.float32)
        npy = os.path.join(self.prior_dir, f"{key}.npy")
        png = os.path.join(self.prior_dir, f"{key}.png")
        if os.path.isfile(npy):
            p = np.load(npy).astype(np.float32)
            p = p/255.0 if p.max() > 1.0 else p
        elif os.path.isfile(png):
            p = np.array(Image.open(png).convert("F"), dtype=np.float32)/255.0
        else:
            raise FileNotFoundError(
                f"Missing prior for '{key}' in {self.prior_dir}")
        if p.shape != (H0, W0):
            p = cv2.resize(
                p, (W0, H0), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        return np.clip(p, 0, 1)

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx) -> Dict[str, object]:
        ipath = self.img_paths[idx]
        name = os.path.splitext(os.path.basename(ipath))[0]
        rgb0 = load_rgb(ipath)
        H0, W0 = rgb0.shape[:2]

        prior0 = self._load_prior(name, H0, W0) if self.use_prior else None

        gt0 = None
        if self.gt_dir:
            # stricter "same-stem" match: name + .<ext>
            cands = [q for q in glob(os.path.join(self.gt_dir, name) + ".*")
                     if os.path.splitext(q)[1].lower() in ALLOWED]
            gt0 = load_mask01(cands[0]) if cands else None

        rgb, _ = resize_keep(rgb0, MAX_SIDE)
        H, W = rgb.shape[:2]

        if self.use_prior:
            prior = cv2.resize(
                prior0, (W, H), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        else:
            prior = np.zeros((H, W), np.float32)

        gt_resized = None
        if gt0 is not None:
            gt_resized = cv2.resize(
                gt0, (W, H), interpolation=cv2.INTER_NEAREST).astype(np.uint8)

        x_rgb = (rgb.astype(np.float32)/255.0).transpose(2, 0, 1)  # [3,H,W]
        x = np.concatenate([x_rgb, prior[None, ...]],
                           0) if self.use_prior else x_rgb

        item = {
            "x": torch.from_numpy(x).float(),
            "rgb_resized": rgb,
            "prior_resized": prior,
            "prior_orig": prior0 if self.use_prior else None,
            "orig_rgb": rgb0,
            "orig_shape": (H0, W0),
            "res_shape": (H, W),
            "name": name
        }
        if gt0 is not None:
            item["gt_orig"] = gt0
            item["gt_resized"] = gt_resized
        return item


def collate_test(b): return b[0]

# =========================
# Model (must match training)
# =========================


def _gn(C):
    for g in (32, 16, 8, 4, 2, 1):
        if C % g == 0:
            return g
    return 1


class Block(nn.Module):
    def __init__(self, c_in, c_out, p=0.0):
        super().__init__()
        g = _gn(c_out)
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
            nn.GroupNorm(g, c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
            nn.GroupNorm(g, c_out), nn.ReLU(inplace=True),
            nn.Dropout2d(p) if p > 0 else nn.Identity()
        )

    def forward(self, x): return self.net(x)


class Down(nn.Module):
    def __init__(self, c_in, c_out, p=0.0):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.block = Block(c_in, c_out, p)

    def forward(self, x): return self.block(self.pool(x))


class Up(nn.Module):
    def __init__(self, c_in, c_skip, c_out, p=0.0):
        super().__init__()
        self.red = nn.Conv2d(c_in, c_out, 1, bias=False)
        self.block = Block(c_out+c_skip, c_out, p)

    def forward(self, x, s):
        x = F.interpolate(x, size=s.shape[-2:],
                          mode="bilinear", align_corners=False)
        x = self.red(x)
        x = torch.cat([x, s], 1)
        return self.block(x)


def _safe_logit(p, eps=1e-6):
    p = p.clamp(eps, 1 - eps)
    return torch.log(p) - torch.log(1 - p)


class UNetTextLines(nn.Module):
    """
    Fused logits = α_map · prior_logits + gate · (prior^beta · RES_SCALE · tanh(delta))
    α_map in [alpha_min, alpha_max] is predicted per-pixel.
    """

    def __init__(self, in_ch=4, base=64, p=0.1,
                 alpha_min=0.15, alpha_max=1.50, beta_gate=1.5, res_scale=2.0):
        super().__init__()
        self.in_ch = in_ch
        self.alpha_min = float(alpha_min)
        self.alpha_max = float(alpha_max)
        self.beta_gate = float(beta_gate)
        self.res_scale = float(res_scale)

        self.e1 = Block(in_ch, base, p)
        self.e2 = Down(base, base*2, p)
        self.e3 = Down(base*2, base*4, p)
        self.e4 = Down(base*4, base*8, p)
        self.bott = Block(base*8, base*16, p)
        self.u4 = Up(base*16, base*8, base*8, p)
        self.u3 = Up(base*8,  base*4, base*4, p)
        self.u2 = Up(base*4,  base*2, base*2, p)
        self.u1 = Up(base*2,  base,   base,   0.0)

        self.head_delta = nn.Conv2d(base, 2, 1)  # residual 2-class logits
        self.head_gate = nn.Conv2d(base, 1, 1)  # residual gate (0,1)
        self.head_alpha = nn.Conv2d(base, 1, 1)  # α-map (0,1) → [αmin, αmax]

        nn.init.zeros_(self.head_delta.weight)
        nn.init.zeros_(self.head_delta.bias)
        nn.init.zeros_(self.head_gate.weight)
        nn.init.constant_(self.head_gate.bias, -2.0)
        nn.init.zeros_(self.head_alpha.weight)
        nn.init.constant_(self.head_alpha.bias, 0.0)

    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        s4 = self.e4(s3)
        z = self.bott(s4)
        z = self.u4(z, s4)
        z = self.u3(z, s3)
        z = self.u2(z, s2)
        z = self.u1(z, s1)

        if self.in_ch >= 4:
            prior = x[:, 3:4].clamp(0, 1)
            l_fg = _safe_logit(prior)
            l_bg = _safe_logit(1.0 - prior)
            prior_logits = torch.cat([l_bg, l_fg], 1)        # [B,2,H,W]
            mask = prior.pow(self.beta_gate)
        else:
            B, _, H, W = x.shape
            prior_logits = torch.zeros(
                B, 2, H, W, device=x.device, dtype=x.dtype)
            mask = 0.0

        delta = self.res_scale * torch.tanh(self.head_delta(z))
        if isinstance(mask, torch.Tensor):
            delta = mask * delta

        gate = torch.sigmoid(self.head_gate(z))
        a01 = torch.sigmoid(self.head_alpha(z))                  # [B,1,H,W]
        alpha_map = self.alpha_min + (self.alpha_max - self.alpha_min) * a01

        fused_logits = alpha_map * prior_logits + gate * \
            delta         # broadcast α over 2 channels
        return fused_logits, alpha_map

# =========================
# Tiled inference
# =========================


def cosine_window_2d(h, w, eps=1e-3):
    wy = np.hanning(h) if h > 1 else np.ones(1)
    wx = np.hanning(w) if w > 1 else np.ones(1)
    w2 = np.outer(wy, wx).astype(np.float32)
    return np.clip(w2, eps, 1.0)


@torch.no_grad()
def tiled_predict_prob_and_alpha(model, x_full, device, patch, stride):
    """
    x_full: [C,H,W] float in [0,1]
    returns: (prob [1,H,W], alpha_avg [1,H,W])
    """
    C, H, W = x_full.shape
    acc_prob = torch.zeros((1, H, W), dtype=torch.float32, device=device)
    wgt_prob = torch.zeros((1, H, W), dtype=torch.float32, device=device)
    acc_alpha = torch.zeros((1, H, W), dtype=torch.float32, device=device)
    wgt_alpha = torch.zeros((1, H, W), dtype=torch.float32, device=device)

    model.eval()
    win_cache: Dict[Tuple[int, int], torch.Tensor] = {}

    ys = list(range(0, max(1, H - patch + 1), stride))
    xs = list(range(0, max(1, W - patch + 1), stride))
    ey = max(0, H - patch)
    ex = max(0, W - patch)
    if ys[-1] != ey:
        ys.append(ey)
    if xs[-1] != ex:
        xs.append(ex)

    it = [(y, x) for y in ys for x in xs]
    it = tqdm(it, desc="Tiling", leave=False) if tqdm else it
    for (y0, x0) in it:
        y1 = min(H, y0 + patch)
        x1 = min(W, x0 + patch)
        tile = x_full[:, y0:y1, x0:x1].unsqueeze(0).to(device)
        tile = tile.contiguous(
            memory_format=torch.channels_last) if device.type == "cuda" else tile
        logits, a_map = model(tile)                     # [1,2,h,w], [1,1,h,w]
        p = torch.softmax(logits, dim=1)[:, 1:2]        # [1,1,h,w]

        h, w = y1 - y0, x1 - x0
        key = (h, w)
        if key not in win_cache:
            win_cache[key] = torch.from_numpy(cosine_window_2d(
                h, w)).to(device=device, dtype=torch.float32)
        win = win_cache[key]

        acc_prob[:,  y0:y1, x0:x1] += p.squeeze(0) * win
        wgt_prob[:,  y0:y1, x0:x1] += win
        acc_alpha[:, y0:y1, x0:x1] += a_map.squeeze(0) * win
        wgt_alpha[:, y0:y1, x0:x1] += win

    prob = torch.where(wgt_prob > 0, acc_prob /
                       wgt_prob,  acc_prob).clamp(0, 1)
    alpha = torch.where(wgt_alpha > 0, acc_alpha / wgt_alpha, acc_alpha)
    return prob, alpha

# Back-compat wrapper if you only need prob


@torch.no_grad()
def tiled_predict_prob(model, x_full, device, patch, stride):
    p, _ = tiled_predict_prob_and_alpha(model, x_full, device, patch, stride)
    return p

# =========================
# Metrics (FULL-IMAGE)
# =========================


def evaluate_metrics_np(gt_u8, pr_u8, thresh=0.75):
    ng, gt_lbl = cv2.connectedComponents(gt_u8)
    np_, pr_lbl = cv2.connectedComponents(pr_u8)
    inter = np.logical_and(gt_lbl > 0, pr_lbl > 0).sum()
    union = np.logical_or(gt_lbl > 0, pr_lbl > 0).sum()
    pixel_IU = inter/union if union > 0 else 0.0
    M, N = ng, np_
    if M <= 1 or N <= 1:
        return pixel_IU, 0.0, 0.0, 0.0, 0.0
    table = np.bincount(gt_lbl.ravel()*N + pr_lbl.ravel(),
                        minlength=M*N).reshape(M, N)
    area_gt = table.sum(1)[:, None]
    area_pr = table.sum(0)[None, :]
    iou = table / (area_gt + area_pr - table + 1e-8)
    sub = iou[1:, 1:]
    best_pr = sub.argmax(1) + 1
    gt_i = np.arange(1, M)
    pr_i = best_pr
    ints = table[gt_i, pr_i]
    precs = ints / (area_pr[0, pr_i] + 1e-8)
    recs = ints / (area_gt[gt_i, 0] + 1e-8)
    CL = (precs >= thresh) & (recs >= thresh)
    ML = (recs < thresh)
    EL = (precs < thresh) & (recs >= thresh)
    line_IU = CL.sum() / max(1, (CL.sum() + ML.sum() + EL.sum()))
    rows, cols = np.where((iou >= thresh) &
                          (np.arange(M)[:, None] > 0) &
                          (np.arange(N)[None, :] > 0))
    matches = len(rows)
    DR = matches / max(1, (M - 1))
    RA = matches / max(1, (N - 1))
    FM = 2 * DR * RA / (DR + RA + 1e-8)
    return pixel_IU, line_IU, DR, RA, FM

# =========================
# Main
# =========================


def main():
    os.makedirs(PRED_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        try:
            torch.set_float32_matmul_precision("high")
        except:
            pass
    print(f"Device: {device}")

    # Load checkpoint
    if not os.path.isfile(CKPT_PATH):
        raise FileNotFoundError(CKPT_PATH)
    state = torch.load(CKPT_PATH, map_location=device)
    sd = state["model"] if isinstance(
        state, dict) and "model" in state else state

    base = int(state.get("base", DEFAULT_BASE))
    in_ch = int(state.get("in_ch", 4))
    patch = int(state.get("patch", DEFAULT_PATCH))
    stride = int(state.get("val_stride", DEFAULT_STRIDE))
    alpha_min = float(state.get("alpha_min", 0.15))
    alpha_max = float(state.get("alpha_max", 1.50))
    beta_gate = float(state.get("beta_gate", 1.5))

    print(f"Model params — base:{base} in_ch:{in_ch} patch:{patch} stride:{stride} "
          f"α∈[{alpha_min},{alpha_max}] β={beta_gate}")

    use_prior = (in_ch >= 4)
    if use_prior and PRIOR_DIR_TEST is None:
        raise RuntimeError(
            "Checkpoint expects prior, but PRIOR_DIR_TEST is None.")

    # Build model (must match training)
    model = UNetTextLines(in_ch=in_ch, base=base, p=0.0,
                          alpha_min=alpha_min, alpha_max=alpha_max, beta_gate=beta_gate).to(device)
    if device.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
    model.load_state_dict(sd, strict=True)
    model.eval()

    # Dataset / loader
    ds = TestDS(TEST_IMG_DIR, PRIOR_DIR_TEST, use_prior, TL_MSK_TEST)
    ld = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_test)

    # Metrics CSV
    csv_path = os.path.join(PRED_DIR, "per_image_metrics.csv")
    import csv
    write_header = not os.path.isfile(csv_path)
    csv_f = open(csv_path, "a", newline="")
    csv_w = csv.writer(csv_f)
    if write_header:
        csv_w.writerow(["name", "H", "W", "thr", "pixel_IU",
                       "line_IU", "DR", "RA", "FM"])

    all_lineiu = []
    all_dice = []

    iterator = tqdm(ld, desc="Inference") if tqdm else ld
    for b in iterator:
        name = b["name"]
        x_full = b["x"].float().to(device)

        H, W = b["res_shape"]
        H0, W0 = b["orig_shape"]
        rgb0 = b["orig_rgb"]

        # Tiled prediction (prob + alpha)
        p_prob, a_full = tiled_predict_prob_and_alpha(
            model, x_full, device, patch, stride)
        prob_res = p_prob[0].cpu().numpy().astype(np.float32)
        alpha_res = a_full[0].cpu().numpy().astype(np.float32)

        # Restore to original size
        prob_up = cv2.resize(prob_res,  (W0, H0),
                             interpolation=cv2.INTER_LINEAR)
        alpha_up = cv2.resize(alpha_res, (W0, H0),
                              interpolation=cv2.INTER_LINEAR)
        mask_bin = (prob_up >= PRED_THR).astype(np.uint8) * 255

        # Save predictions
        if SAVE_PROB:
            Image.fromarray(np.clip(prob_up*255.0, 0, 255).astype(np.uint8)).save(
                os.path.join(PRED_DIR, f"{name}_prob.png"))
        if SAVE_NPY:
            np.save(os.path.join(PRED_DIR, f"{name}_prob.npy"), prob_up)
        Image.fromarray(mask_bin).save(os.path.join(
            PRED_DIR, f"{name}_mask_thr{int(PRED_THR*100)}.png"))

        if SAVE_OVERLAY:
            prob_u8 = np.clip(prob_up*255.0, 0, 255).astype(np.uint8)
            overlay = cv2.addWeighted(cv2.cvtColor(rgb0, cv2.COLOR_RGB2BGR), 0.45,
                                      cv2.applyColorMap(prob_u8, cv2.COLORMAP_JET), 0.55, 0)
            cv2.imwrite(os.path.join(PRED_DIR, f"{name}_overlay.png"), overlay)

        # Optional: save α-map for debugging how strong the prior was
        if SAVE_ALPHA:
            a_u8 = np.clip(alpha_up / max(1e-8, alpha_max)
                           * 255.0, 0, 255).astype(np.uint8)
            Image.fromarray(a_u8).save(
                os.path.join(PRED_DIR, f"{name}_alpha.png"))
            a_ov = cv2.addWeighted(cv2.cvtColor(rgb0, cv2.COLOR_RGB2BGR), 0.45,
                                   cv2.applyColorMap(a_u8, cv2.COLORMAP_VIRIDIS), 0.55, 0)
            cv2.imwrite(os.path.join(
                PRED_DIR, f"{name}_alpha_overlay.png"), a_ov)

        # Save prior for reference
        if use_prior and SAVE_PRIOR and b["prior_orig"] is not None:
            prior0 = b["prior_orig"].astype(np.float32)
            prior_u8 = np.clip(prior0*255.0, 0, 255).astype(np.uint8)
            Image.fromarray(prior_u8).save(
                os.path.join(PRED_DIR, f"{name}_prior.png"))
            if SAVE_PRIOR_OVERLAY:
                overlay_p = cv2.addWeighted(cv2.cvtColor(rgb0, cv2.COLOR_RGB2BGR), 0.45,
                                            cv2.applyColorMap(prior_u8, cv2.COLORMAP_JET), 0.55, 0)
                cv2.imwrite(os.path.join(
                    PRED_DIR, f"{name}_prior_overlay.png"), overlay_p)

        # Metrics (if GT present)
        if "gt_orig" in b:
            gt0 = (b["gt_orig"].astype(np.uint8) * 255)
            pr0 = mask_bin
            pixel_IU, line_IU, DR, RA, FM = evaluate_metrics_np(
                gt0, pr0, thresh=0.75)

            gg = (gt0 > 0).astype(np.uint8)
            pp = (pr0 > 0).astype(np.uint8)
            tp = (gg & pp).sum()
            fp = ((pp == 1) & (gg == 0)).sum()
            fn = ((pp == 0) & (gg == 1)).sum()
            dice = (2*tp) / (2*tp + fp + fn + 1e-8)

            all_lineiu.append(line_IU)
            all_dice.append(dice)

            print(f"[{name}] size={H0}x{W0}  thr={PRED_THR:.2f}  "
                  f"PixelIoU={pixel_IU:.4f}  LineIU={line_IU:.4f}  DR={DR:.4f}  RA={RA:.4f}  FM={FM:.4f}  Dice={dice:.4f}")
            csv_w.writerow(
                [name, H0, W0, PRED_THR, pixel_IU, line_IU, DR, RA, FM])
        else:
            print(f"[{name}] saved (no GT -> metrics skipped)")

    csv_f.close()

    if all_lineiu:
        print("\n=== SUMMARY (images with GT) ===")
        print(f"Mean LineIU: {float(np.mean(all_lineiu)):.4f}")
        print(f"Mean Dice  : {float(np.mean(all_dice)):.4f}")
        print(f"CSV: {csv_path}")
    print(f"Predictions saved to: {PRED_DIR}")


# =========================
# ENTRY
# =========================
if __name__ == "__main__":
    main()

Device: cuda
Model params — base:64 in_ch:4 patch:512 stride:256 α∈[0.15,1.5] β=1.5
[test] 43 images from /kaggle/input/udids2025/UDIADS/images/test
[test] Using priors from: /kaggle/working//priors/test
[test] Will compute metrics using GT from: /kaggle/input/udids2025/UDIADS/masks/test


Inference:   2%|▏         | 1/43 [00:04<02:49,  4.02s/it]A

[031] size=2016x1344  thr=0.50  PixelIoU=0.7684  LineIU=0.8610  DR=0.5989  RA=0.2667  FM=0.3690  Dice=0.8690



Inference:   5%|▍         | 2/43 [00:07<02:41,  3.94s/it]A

[032] size=2016x1344  thr=0.50  PixelIoU=0.8689  LineIU=0.9429  DR=0.9143  RA=0.5333  FM=0.6737  Dice=0.9298


...

[313] size=2016x1344  thr=0.50  PixelIoU=0.7945  LineIU=0.9180  DR=0.8306  RA=0.3689  FM=0.5109  Dice=0.8855



Inference:  98%|█████████▊| 42/43 [02:44<00:03,  3.94s/it]

[362] size=2016x1344  thr=0.50  PixelIoU=0.8080  LineIU=0.9079  DR=0.8816  RA=0.3941  FM=0.5447  Dice=0.8938



Inference: 100%|██████████| 43/43 [02:48<00:00,  3.92s/it]

[368] size=2016x1344  thr=0.50  PixelIoU=0.8193  LineIU=0.9351  DR=0.8896  RA=0.4893  FM=0.6313  Dice=0.9007

=== SUMMARY (images with GT) ===
Mean LineIU: 0.9430
Mean Dice  : 0.9110
CSV: /kaggle/working/Textlines/preds/per_image_metrics.csv
Predictions saved to: /kaggle/working/Textlines/preds


In [8]:
import os, shutil
from IPython.display import FileLink

src = "/kaggle/working/Textlines/preds"
zip_base = "/kaggle/working/preds"          # will create /kaggle/working/preds.zip
zip_path = zip_base + ".zip"

# remove old zip if it exists (optional)
if os.path.exists(zip_path):
    os.remove(zip_path)

# create zip
shutil.make_archive(zip_base, "zip", src)

# clickable link to download
FileLink(zip_path)

/kaggle/working/preds.zip